<a href="https://colab.research.google.com/github/kshelp/pytorchtrf/blob/main/09%EC%9E%A5%20%EA%B0%9D%EC%B2%B4%20%ED%83%90%EC%A7%80/%EC%BB%A4%EC%8A%A4%ED%85%80_%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%A1%9C_Segmentation_%ED%95%99%EC%8A%B5%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
######################
# 0. 코랩 환경 준비
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-seg.pt")   #

results = model.train(
    #data   = "../dataset_yolo_seg/dataset.yaml", # dataset.yaml 파일 경로
    data   = "/content/drive/MyDrive/dataset_yolo_seg/dataset.yaml", # dataset.yaml 파일 경로
    epochs = 100,   # 훈련 에폭 수
    imgsz  = 640,   # 이미지 크기
    batch  = 16,    # 배치 크기
    device = "cpu"   # GPU index 0, or "cpu"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/segment/train/weights/best.pt")

metrics = model.val()

print(f"Box  mAP50-95 : {metrics.box.map:.4f}")
print(f"Mask mAP50-95 : {metrics.seg.map:.4f}")
print(f"Mask mAP50    : {metrics.seg.map50:.4f}")

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("runs/segment/train/weights/best.pt")

cap = cv2.VideoCapture(0)

if cap.isOpened():
    print("실시간 세그멘테이션 시작")
    print("종료하려면 'q'를 누르세요.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("프레임을 읽을 수 없습니다.")
            break

        # 추론
        results = model(frame, conf=0.25, iou=0.45, verbose=False)

        # 결과가 그려진 프레임
        annotated_frame = results[0].plot()

        # 화면 출력
        cv2.imshow("YOLO Segmentation Realtime", annotated_frame)

        # q 누르면 종료
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()